In [11]:
import test

from spherinator.data import DataModule, Column

loader = DataModule(
    path="/home/bernd/data/SKIRT_synthetic_images_small/parquet-128",
    columns=[Column("data")],
    return_dict=False,
    validation_size=0.2,
    test_size=0.0,
    batch_size=10,
    shuffle=True,
    num_workers=0,  # must be 0 when debugging; debugpy hangs on forked worker processes
)
loader.setup()

In [12]:
train_dataloader = loader.train_dataloader()
print(len(train_dataloader))
data = next(iter(train_dataloader))
print(data.shape)

val_dataloader = loader.val_dataloader()
print(len(val_dataloader))

test_dataloader = loader.test_dataloader()
print(len(test_dataloader))

16
torch.Size([10, 3, 128, 128])
4
4


In [7]:
from datasets import load_dataset

full_ds = load_dataset("/home/bernd/data/SKIRT_synthetic_images_small/parquet-128", split="train")
split_ds = full_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split_ds["train"]

# Split the 20% into two 10% chunks for Val and Test
val_test_ds = split_ds["test"].train_test_split(test_size=0.5, seed=42)

train_ds = train_ds
val_ds = val_test_ds["train"]
test_ds = val_test_ds["test"]
print(train_ds)
print(len(train_ds))
print(train_ds.features["data"])
print(train_ds._data.schema.metadata)
print(val_ds)
print(len(val_ds))

Dataset({
    features: ['data', 'simulation', 'snapshot', 'subhalo_id'],
    num_rows: 160
})
160
List(Value('float64'))
{b'huggingface': b'{"info": {"features": {"data": {"feature": {"dtype": "float64", "_type": "Value"}, "_type": "List"}, "simulation": {"dtype": "string", "_type": "Value"}, "snapshot": {"dtype": "string", "_type": "Value"}, "subhalo_id": {"dtype": "string", "_type": "Value"}}}}'}
Dataset({
    features: ['data', 'simulation', 'snapshot', 'subhalo_id'],
    num_rows: 20
})
20


In [8]:
ds = load_dataset("/home/bernd/data/SKIRT_synthetic_images_small/parquet-128", split="train", streaming=True)